In [ ]:
# CODE FOR: 2_create_master.ipynb
import pandas as pd
import joblib
import re

# Load Brains
model = joblib.load("../models/inspector_model.pkl")
vectorizer = joblib.load("../models/tfidf_vectorizer.pkl")

# Load Silver
df = pd.read_csv("../data/pakwheels_silver_data.csv")

# 1. Clean Numbers (The "Managed by Pakwheels" Fix)
def clean_nums(x):
    if not isinstance(x, str): return None
    if "\n" in x: x = x.split("\n")[0]
    x = x.lower().replace(',', '').replace('km', '').replace('cc', '').replace('pkr', '').strip()
    if 'crore' in x: return float(x.replace('crore', '').strip()) * 10000000
    if 'lacs' in x: return float(x.replace('lacs', '').strip()) * 100000
    if x.replace('.', '', 1).isdigit(): return float(x)
    return None

for col in ['price', 'mileage', 'engine']:
    df[col] = df[col].apply(clean_nums)

df = df.dropna(subset=['price', 'mileage', 'engine'])

# 2. Grade Silver Cars (Using Smart Tagger)
# (Copy the smart_condition_tagger function from 1 train inspector here if not in memory, 
# but usually it's not needed for prediction if the vectorizer handles tokens, 
# HOWEVER, to be safe, we apply it.)
def smart_condition_tagger(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    if ("total genuine" in text or "sealed" in text or "no touch" in text): return text + " TAG_EXCELLENT"
    minor_pattern = r"\b(1|one|2|two|3|three)(\s+(or|to|-)\s+(2|two|3|three))?\s*(piece|pc|touch)"
    if re.search(minor_pattern, text) or "minor" in text: return text + " TAG_GOOD"
    if "shower" in text or "repaint" in text: return text + " TAG_FAIR"
    return text

df['description'] = df['description'].fillna('').apply(smart_condition_tagger)
X_silver = vectorizer.transform(df['description'])
df['inspection_score'] = model.predict(X_silver).round(1)
df['data_source'] = 'silver_predicted'

# 3. Load Gold & Merge
gold = pd.read_csv("../data/gold_data_CLEANED.csv")
gold['data_source'] = 'gold_verified'

# Align Columns (The Fix)
gold = gold.rename(columns={'price_cleaned': 'price', 'mileage_cleaned': 'mileage', 'engine_cleaned': 'engine'})
master = pd.concat([gold, df], ignore_index=True)

# Save
cols = ['title_version', 'model_year', 'mileage', 'engine', 'transmission', 'fuel', 'price', 'inspection_score', 'description', 'data_source']
master = master[cols] # Keep only clean columns
master.to_csv("../data/MASTER_CAR_DATASET.csv", index=False)
print(f"✅ Step 2 Complete: Master Dataset Created ({len(master)} cars).")

✅ Step 2 Complete: Master Dataset Created (3808 cars).
